# 🤖 AI Customer Review Intelligence Dashboard
### Advanced NLP Sentiment Analysis | Amazon Beauty Products
---
**Features:** Multi-model BERT analysis · 13+ interactive charts · Dynamic widget UI · Word clouds · Heatmaps · Trends


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — INSTALLATION (Run once, then restart runtime)     ║
# ╚══════════════════════════════════════════════════════════════╝
import subprocess, sys

pkgs = [
    "transformers", "torch", "datasets", "wordcloud",
    "plotly", "ipywidgets", "pyarrow", "pandas",
    "numpy", "matplotlib", "seaborn", "tqdm", "kaleido"
]

for pkg in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Enable ipywidgets in Colab
from google.colab import output
output.enable_custom_widget_manager()

print("\n✅ All packages installed! Proceed to the next cell.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — IMPORTS & DASHBOARD STYLING                       ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from transformers import pipeline
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import matplotlib
from collections import Counter
from datetime import datetime, timedelta
import re, warnings, textwrap
warnings.filterwarnings("ignore")

# ── Professional CSS for Google Colab UI ──────────────────────
display(HTML("""
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700;800&family=DM+Mono:wght@400;500&display=swap" rel="stylesheet">

<style>
  /* ---- Base ---- */
  body, .widget-label, .widget-readout { font-family: 'DM Sans', sans-serif !important; }

  /* ---- Hero Header ---- */
  .ai-header {
    background: linear-gradient(135deg, #0f0c29, #302b63, #24243e);
    padding: 36px 40px;
    border-radius: 18px;
    margin: 8px 0 20px;
    position: relative;
    overflow: hidden;
    box-shadow: 0 20px 60px rgba(0,0,0,0.4);
  }
  .ai-header::before {
    content: '';
    position: absolute;
    top: -60px; right: -60px;
    width: 300px; height: 300px;
    background: radial-gradient(circle, rgba(102,126,234,0.35) 0%, transparent 70%);
    border-radius: 50%;
  }
  .ai-header h1 {
    font-family: 'DM Sans', sans-serif;
    color: white; font-size: 2em; font-weight: 800;
    margin: 0 0 8px; letter-spacing: -0.5px;
  }
  .ai-header p { color: rgba(255,255,255,0.65); margin: 0; font-size: 0.95em; }
  .ai-header .badge {
    display: inline-block; background: rgba(102,126,234,0.4);
    color: #a5b4fc; padding: 3px 10px; border-radius: 20px;
    font-size: 0.75em; font-weight: 600; margin-right: 6px;
    border: 1px solid rgba(165,180,252,0.3);
  }

  /* ---- Metric Cards ---- */
  .metrics-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(130px, 1fr));
    gap: 12px; margin: 16px 0;
  }
  .metric-card {
    background: white;
    border-radius: 14px; padding: 18px 16px;
    text-align: center;
    box-shadow: 0 4px 20px rgba(0,0,0,0.06);
    border-top: 3px solid transparent;
    transition: all 0.25s ease;
  }
  .metric-card:hover { transform: translateY(-4px); box-shadow: 0 8px 30px rgba(0,0,0,0.1); }
  .metric-card.blue  { border-color: #667eea; }
  .metric-card.green { border-color: #00C897; }
  .metric-card.red   { border-color: #FF4B6E; }
  .metric-card.amber { border-color: #FFC107; }
  .metric-val { font-size: 1.9em; font-weight: 800; color: #1a1a2e; line-height: 1; }
  .metric-lbl { font-size: 0.72em; color: #888; text-transform: uppercase;
                letter-spacing: 0.8px; margin-top: 5px; font-weight: 600; }

  /* ---- Panel Containers ---- */
  .panel {
    background: #fafbff;
    border: 1px solid #e8eaf0;
    border-radius: 14px; padding: 22px 24px; margin: 10px 0;
  }
  .panel-title {
    font-size: 1.05em; font-weight: 700; color: #1a1a2e;
    margin-bottom: 16px; display: flex; align-items: center; gap: 8px;
  }
  .panel-title span { font-size: 1.2em; }

  /* ---- Status Pills ---- */
  .pill { display:inline-block; padding:4px 14px; border-radius:20px;
          font-size:0.78em; font-weight:700; letter-spacing:0.3px; }
  .pill-pos { background:#d1fae5; color:#065f46; }
  .pill-neg { background:#ffe4e6; color:#9f1239; }
  .pill-neu { background:#fef3c7; color:#92400e; }
  .pill-info { background:#dbeafe; color:#1e40af; }

  /* ---- Results Table ---- */
  .results-table { width:100%; border-collapse:collapse; font-size:0.85em; }
  .results-table th { background:#f1f3ff; padding:10px 14px; text-align:left;
                      font-weight:700; color:#444; border-bottom:2px solid #e0e4f5; }
  .results-table td { padding:9px 14px; border-bottom:1px solid #f0f1f8; color:#333; }
  .results-table tr:hover td { background:#f8f9ff; }

  /* ── Force widget label font ── */
  .widget-label-basic { font-family:'DM Sans', sans-serif !important; font-size:0.9em !important; }
</style>
"""))

display(HTML("""
<div class="ai-header">
  <h1>🤖 AI Review Intelligence Dashboard</h1>
  <p style="margin-bottom:12px">Advanced NLP-Powered Customer Sentiment & Emotion Analysis</p>
  <span class="badge">🧠 BERT Models</span>
  <span class="badge">📊 13 Chart Types</span>
  <span class="badge">⚡ Real-time Analysis</span>
  <span class="badge">🎨 Interactive UI</span>
</div>
"""))

print("✅  Imports & styling loaded!")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — DATA LOADING & PREPARATION                        ║
# ╚══════════════════════════════════════════════════════════════╝

def load_amazon_data(n_rows=300):
    """Load Amazon Beauty Products dataset with fallback to synthetic data."""
    print("📦  Loading Amazon Beauty Products dataset…")

    url = ("https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023"
           "/resolve/main/raw_meta_All_Beauty/full-00000-of-00001.parquet")
    try:
        raw = pd.read_parquet(url, engine="pyarrow").iloc[:n_rows]
        raw["review_text"] = raw["description"].apply(
            lambda x: x[0] if isinstance(x, (list, np.ndarray)) and len(x) > 0 else str(x)
        )
        # Strip HTML tags and short strings
        raw["review_text"] = raw["review_text"].str.replace(r"<[^>]+>", "", regex=True).str.strip()
        raw["review_text"] = raw["review_text"].apply(
            lambda t: t if len(str(t)) > 20 else "Standard beauty product with typical features."
        )
        df = raw[["review_text", "average_rating", "title", "main_category"]].copy()
        df.rename(columns={"average_rating": "rating", "title": "product_title"}, inplace=True)
        source = "Amazon dataset"
    except Exception as err:
        print(f"  ⚠️  Online fetch failed ({str(err)[:60]}). Using rich synthetic dataset…")

        review_templates = [
            ("This moisturizer is absolutely life-changing! Skin is glowing and hydrated.", 5, "Skincare"),
            ("The fragrance is divine and incredibly long-lasting. Pure luxury!", 5, "Fragrance"),
            ("Terrible product. Caused a severe allergic reaction. Avoid completely.", 1, "Skincare"),
            ("Decent shampoo but nothing groundbreaking. Hair feels clean at least.", 3, "Haircare"),
            ("This foundation gives flawless coverage that lasts the entire day!", 5, "Makeup"),
            ("Absolutely disgusting smell. Gave me a splitting headache immediately.", 1, "Fragrance"),
            ("Good value for what you pay. Works as advertised, nothing more.", 3, "General"),
            ("This anti-aging serum reduced my fine lines visibly within 2 weeks!", 5, "Skincare"),
            ("Cute packaging but the formula is watery and ineffective. Waste.", 2, "Skincare"),
            ("Best SPF sunscreen I have ever used — lightweight and invisible!", 5, "Suncare"),
            ("Lipstick formula is creamy and the color stays on for 8+ hours.", 5, "Makeup"),
            ("Hair dye turned my hair orange instead of the advertised blonde.", 1, "Haircare"),
            ("Reasonable cleanser for daily use. Skin doesn't feel stripped.", 4, "Skincare"),
            ("The nail polish chips after just one day. Extremely disappointing.", 2, "Nails"),
            ("I've been using this eye cream for a month and dark circles are gone!", 5, "Skincare"),
            ("Blush pigmentation is stunning. A tiny amount goes a very long way.", 4, "Makeup"),
            ("Conditioner made my hair incredibly soft and manageable. Love it!", 5, "Haircare"),
            ("This toner irritated my sensitive skin terribly. Red and burning.", 1, "Skincare"),
            ("Perfume bottle broke during shipping. Poor packaging from seller.", 2, "Fragrance"),
            ("Body lotion has a wonderful scent and absorbs quickly without grease.", 4, "Bodycare"),
        ]

        rows = []
        np.random.seed(42)
        for i in range(n_rows):
            t = review_templates[i % len(review_templates)]
            rows.append({
                "review_text": t[0],
                "rating": t[1],
                "product_title": f"{t[2]} Product #{i+1}",
                "main_category": t[2],
            })
        df = pd.DataFrame(rows)
        source = "synthetic dataset"

    # Synthesise dates (last 120 days) and derived columns
    np.random.seed(42)
    df["date"] = [datetime.now() - timedelta(days=int(np.random.randint(0, 120)))
                  for _ in range(len(df))]
    df["week"]  = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["word_count"] = df["review_text"].str.split().str.len()

    print(f"  ✅  Loaded {len(df)} records from {source}.")
    return df


# ── Load and preview ──────────────────────────────────────────
RAW_DF = load_amazon_data(300)

display(HTML(f"""
<div class="metrics-grid">
  <div class="metric-card blue">
    <div class="metric-val">{len(RAW_DF)}</div>
    <div class="metric-lbl">Total Records</div>
  </div>
  <div class="metric-card green">
    <div class="metric-val">{RAW_DF["rating"].mean():.1f}★</div>
    <div class="metric-lbl">Avg Rating</div>
  </div>
  <div class="metric-card amber">
    <div class="metric-val">{RAW_DF["main_category"].nunique()}</div>
    <div class="metric-lbl">Categories</div>
  </div>
  <div class="metric-card red">
    <div class="metric-val">{RAW_DF["word_count"].mean():.0f}</div>
    <div class="metric-lbl">Avg Words</div>
  </div>
</div>
"""))

RAW_DF[["product_title", "main_category", "rating", "review_text", "date"]].head(8)


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — AI ANALYSIS ENGINE (Multi-model NLP)              ║
# ╚══════════════════════════════════════════════════════════════╝

class SentimentEngine:
    """
    Multi-model NLP analysis engine.
    Supports four transformer models with automatic label normalisation.
    """

    MODELS = {
        "⚡ DistilBERT — Fast Binary":       "distilbert-base-uncased-finetuned-sst-2-english",
        "🎭 Emotion Detector (6 classes)":    "j-hartmann/emotion-english-distilroberta-base",
        "⭐ 5-Star Rating Predictor":          "nlptown/bert-base-multilingual-uncased-sentiment",
        "🐦 RoBERTa Twitter Sentiment":        "cardiffnlp/twitter-roberta-base-sentiment-latest",
    }

    # Unified label normalisation map
    _LABEL_MAP = {
        "POSITIVE": "POSITIVE", "NEGATIVE": "NEGATIVE", "NEUTRAL": "NEUTRAL",
        "POS": "POSITIVE", "NEG": "NEGATIVE", "NEU": "NEUTRAL",
        "LABEL_2": "POSITIVE", "LABEL_1": "NEUTRAL", "LABEL_0": "NEGATIVE",
        "5 stars": "POSITIVE", "4 stars": "POSITIVE",
        "3 stars": "NEUTRAL",
        "2 stars": "NEGATIVE", "1 star": "NEGATIVE",
        "joy": "POSITIVE", "surprise": "POSITIVE",
        "sadness": "NEGATIVE", "anger": "NEGATIVE",
        "fear": "NEGATIVE", "disgust": "NEGATIVE",
    }

    def __init__(self):
        self._pipeline   = None
        self._loaded_key = None

    def _load(self, model_key: str):
        if self._loaded_key == model_key:
            return
        print(f"  🧠  Loading model: {model_key} …")
        self._pipeline   = pipeline("text-classification",
                                    model=self.MODELS[model_key],
                                    truncation=True, max_length=512)
        self._loaded_key = model_key
        print("  ✅  Model ready!")

    def analyse(self, df: pd.DataFrame, model_key: str, batch_size: int = 16) -> pd.DataFrame:
        self._load(model_key)
        texts   = df["review_text"].tolist()
        results = []
        total   = len(texts)

        for i in range(0, total, batch_size):
            batch   = texts[i : i + batch_size]
            results.extend(self._pipeline(batch, truncation=True))
            pct = min(i + batch_size, total) / total * 100
            print(f"  ⏳  {pct:.0f}%  ({min(i+batch_size, total)}/{total})", end="\r")

        print(f"  ✅  Analysed {total} texts.                  ")

        out = df.copy()
        out["raw_label"]  = [r["label"] for r in results]
        out["confidence"] = [r["score"]  for r in results]
        out["sentiment"]  = out["raw_label"].map(
            lambda lbl: self._LABEL_MAP.get(lbl, "NEUTRAL")
        )
        out["score"] = out["sentiment"].map(
            {"POSITIVE": 1, "NEUTRAL": 0, "NEGATIVE": -1}
        )
        return out


ENGINE = SentimentEngine()
print("✅  SentimentEngine initialised — ready for analysis.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — VISUALISATION ENGINE (13 Chart Types)             ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Shared Plotly theme ───────────────────────────────────────
_TEMPLATE  = "plotly_white"
_C_POS     = "#00C897"
_C_NEG     = "#FF4B6E"
_C_NEU     = "#FFC107"
_C_MAP     = {"POSITIVE": _C_POS, "NEGATIVE": _C_NEG, "NEUTRAL": _C_NEU}
_C_PRIMARY = "#667eea"
_FONT      = dict(family="DM Sans, Google Sans, sans-serif", color="#1a1a2e")
_LAYOUT    = dict(template=_TEMPLATE, font=_FONT,
                  paper_bgcolor="white", plot_bgcolor="#f9faff",
                  margin=dict(t=60, b=40, l=40, r=40),
                  title_font=dict(size=17, color="#1a1a2e"))


class VizEngine:
    """Creates all 13 chart types from a results DataFrame."""

    def __init__(self, df: pd.DataFrame):
        self.df = df

    # ─────────────────────────── 1 ──────────────────────────────
    def sentiment_donut(self) -> go.Figure:
        cnt = self.df["sentiment"].value_counts()
        colors = [_C_MAP.get(s, "#aaa") for s in cnt.index]
        pos_pct = cnt.get("POSITIVE", 0) / len(self.df) * 100

        fig = go.Figure(go.Pie(
            labels=cnt.index, values=cnt.values,
            hole=0.58, marker_colors=colors,
            textinfo="label+percent", textfont_size=12,
            pull=[0.06 if s == cnt.idxmax() else 0 for s in cnt.index],
            hovertemplate="%{label}: %{value} reviews<extra></extra>",
        ))
        fig.add_annotation(text=f"<b>{pos_pct:.0f}%</b><br><span style='font-size:12px'>Positive</span>",
                           x=0.5, y=0.5, showarrow=False, font=dict(size=20, color="#1a1a2e"))
        fig.update_layout(**_LAYOUT, title="Overall Sentiment Distribution", height=420, showlegend=True)
        return fig

    # ─────────────────────────── 2 ──────────────────────────────
    def rating_bar(self) -> go.Figure:
        rc = self.df["rating"].round().astype(int).value_counts().sort_index()
        pal = ["#FF4B6E", "#FF8C42", "#FFC107", "#7CB9E8", "#00C897"]
        fig = go.Figure(go.Bar(
            x=[f"★ {r}" for r in rc.index], y=rc.values,
            marker_color=pal[:len(rc)],
            text=rc.values, textposition="outside",
            hovertemplate="Rating %{x}: %{y} reviews<extra></extra>",
        ))
        fig.update_layout(**_LAYOUT, title="Star Rating Distribution",
                          xaxis_title="Star Rating", yaxis_title="Count",
                          height=420, showlegend=False)
        return fig

    # ─────────────────────────── 3 ──────────────────────────────
    def sentiment_trend(self) -> go.Figure:
        dfs = self.df.sort_values("date").copy()
        dfs["ma10"] = dfs["score"].rolling(10, min_periods=1).mean()
        fig = go.Figure()
        for sent, col in _C_MAP.items():
            m = dfs["sentiment"] == sent
            fig.add_trace(go.Scatter(
                x=dfs[m]["date"], y=dfs[m]["score"],
                mode="markers", name=sent,
                marker=dict(color=col, size=7, opacity=0.55),
            ))
        fig.add_trace(go.Scatter(
            x=dfs["date"], y=dfs["ma10"],
            mode="lines", name="10-item Moving Avg",
            line=dict(color=_C_PRIMARY, width=3),
        ))
        fig.add_hline(y=0, line_dash="dash", line_color="#aaa", opacity=0.6)
        fig.update_layout(**_LAYOUT, title="Sentiment Trend Over Time",
                          xaxis_title="Date", yaxis_title="Sentiment Score",
                          height=430, hovermode="x unified")
        return fig

    # ─────────────────────────── 4 ──────────────────────────────
    def confidence_histogram(self) -> go.Figure:
        fig = px.histogram(
            self.df, x="confidence", color="sentiment",
            nbins=30, color_discrete_map=_C_MAP,
            barmode="overlay", opacity=0.72,
            title="Model Confidence Score Distribution",
        )
        fig.update_layout(**_LAYOUT, xaxis_title="Confidence", yaxis_title="Count", height=420)
        return fig

    # ─────────────────────────── 5 ──────────────────────────────
    def category_grouped_bar(self) -> go.Figure:
        grp = self.df.groupby(["main_category", "sentiment"]).size().reset_index(name="count")
        fig = px.bar(grp, x="main_category", y="count", color="sentiment",
                     barmode="group", color_discrete_map=_C_MAP,
                     title="Sentiment Breakdown by Product Category")
        fig.update_layout(**_LAYOUT, xaxis_title="Category", yaxis_title="Count",
                          height=440, xaxis_tickangle=-30)
        return fig

    # ─────────────────────────── 6 ──────────────────────────────
    def rating_sentiment_heatmap(self) -> go.Figure:
        heat = pd.crosstab(self.df["rating"].round().astype(int), self.df["sentiment"])
        fig  = px.imshow(heat, color_continuous_scale="RdYlGn",
                         title="Rating ↔ Sentiment Heatmap", text_auto=True, aspect="auto")
        fig.update_layout(**_LAYOUT, xaxis_title="Sentiment", yaxis_title="Star Rating", height=430)
        return fig

    # ─────────────────────────── 7 ──────────────────────────────
    def top_keywords(self, n: int = 20) -> go.Figure:
        stops = {
            "the","a","an","and","or","but","in","on","at","to","for","of",
            "with","is","it","this","that","was","are","be","as","i","my",
            "your","not","have","had","has","by","from","its","will","very",
            "so","if","no","can","do","up","out","about","just","been","also",
            "get","more","its","than","too","use","used","using","got","get",
        }
        words = []
        for txt in self.df["review_text"]:
            words.extend([w for w in re.findall(r"\b[a-z]{3,}\b", txt.lower()) if w not in stops])
        wc_top = Counter(words).most_common(n)
        ws, cs = zip(*wc_top) if wc_top else ([], [])
        fig = go.Figure(go.Bar(
            x=list(cs), y=list(ws), orientation="h",
            marker=dict(color=list(cs), colorscale="Viridis", showscale=False),
            text=list(cs), textposition="outside",
        ))
        fig.update_layout(**_LAYOUT, title=f"Top {n} Most Frequent Keywords",
                          xaxis_title="Frequency", yaxis=dict(autorange="reversed"),
                          height=520, showlegend=False)
        return fig

    # ─────────────────────────── 8 ──────────────────────────────
    def word_cloud_fig(self, sentiment_filter: str = "ALL") -> plt.Figure:
        sub = (self.df[self.df["sentiment"] == sentiment_filter]["review_text"]
               if sentiment_filter != "ALL" else self.df["review_text"])
        text = " ".join(sub) or "no data"
        cmap_map = {"ALL": "magma", "POSITIVE": "Greens", "NEGATIVE": "Reds", "NEUTRAL": "Blues"}
        wc = WordCloud(background_color="white", colormap=cmap_map.get(sentiment_filter, "magma"),
                       max_words=80, width=900, height=400,
                       prefer_horizontal=0.88, collocations=False).generate(text)
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
        ax.set_title(f"Word Cloud — {sentiment_filter} Reviews",
                     fontsize=16, fontweight="bold", pad=15, color="#1a1a2e")
        plt.tight_layout()
        return fig

    # ─────────────────────────── 9 ──────────────────────────────
    def sentiment_gauge(self) -> go.Figure:
        score = self.df["score"].mean()
        fig = go.Figure(go.Indicator(
            mode="gauge+number+delta",
            value=round(score, 3),
            delta={"reference": 0, "valueformat": ".3f"},
            title={"text": "Overall Sentiment Score<br><sub>-1.0 (Very Negative) → +1.0 (Very Positive)</sub>",
                   "font": {"size": 16, "color": "#1a1a2e"}},
            gauge={
                "axis": {"range": [-1, 1], "tickwidth": 1, "tickcolor": "#aaa"},
                "bar": {"color": _C_PRIMARY, "thickness": 0.28},
                "bgcolor": "#f9faff", "borderwidth": 1, "bordercolor": "#e0e0e0",
                "steps": [
                    {"range": [-1, -0.33], "color": "#ffe4e6"},
                    {"range": [-0.33, 0.33], "color": "#fef9c3"},
                    {"range": [0.33, 1],    "color": "#dcfce7"},
                ],
                "threshold": {"line": {"color": "#333", "width": 3},
                               "thickness": 0.75, "value": score},
            },
        ))
        fig.update_layout(**_LAYOUT, height=370)
        return fig

    # ─────────────────────────── 10 ─────────────────────────────
    def confidence_boxplot(self) -> go.Figure:
        fig = px.box(self.df, x="sentiment", y="confidence",
                     color="sentiment", color_discrete_map=_C_MAP,
                     title="Confidence Distribution by Sentiment",
                     points="all", notched=True)
        fig.update_layout(**_LAYOUT, height=430, showlegend=False,
                          xaxis_title="Sentiment", yaxis_title="Confidence Score")
        return fig

    # ─────────────────────────── 11 ─────────────────────────────
    def scatter_rating_confidence(self) -> go.Figure:
        fig = px.scatter(
            self.df, x="rating", y="confidence", color="sentiment",
            color_discrete_map=_C_MAP, opacity=0.65,
            title="Star Rating vs Model Confidence",
            trendline="lowess",
            hover_data={"review_text": ":.40s"},
        )
        fig.update_layout(**_LAYOUT, height=430,
                          xaxis_title="Star Rating", yaxis_title="Confidence Score")
        return fig

    # ─────────────────────────── 12 ─────────────────────────────
    def sunburst_category_sentiment(self) -> go.Figure:
        grp = self.df.groupby(["main_category","sentiment"]).size().reset_index(name="count")
        fig = px.sunburst(grp, path=["main_category","sentiment"], values="count",
                          color="sentiment", color_discrete_map=_C_MAP,
                          title="Category → Sentiment Hierarchy (Sunburst)")
        fig.update_layout(**_LAYOUT, height=500)
        return fig

    # ─────────────────────────── 13 ─────────────────────────────
    def word_count_violin(self) -> go.Figure:
        fig = px.violin(self.df, x="sentiment", y="word_count",
                        color="sentiment", color_discrete_map=_C_MAP,
                        box=True, points="all", title="Review Word Count by Sentiment")
        fig.update_layout(**_LAYOUT, height=430, showlegend=False,
                          xaxis_title="Sentiment", yaxis_title="Word Count")
        return fig


print("✅  VizEngine loaded — 13 chart types ready.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — INTERACTIVE DASHBOARD  (Run this cell)            ║
# ╚══════════════════════════════════════════════════════════════╝

class Dashboard:
    """Full interactive dashboard — select model, run analysis, explore any chart."""

    CHARTS = {
        "📊  Sentiment Donut":             "sentiment_donut",
        "⭐  Rating Distribution":          "rating_bar",
        "📈  Sentiment Trend":             "sentiment_trend",
        "🎯  Confidence Histogram":        "confidence_histogram",
        "🗂️  Sentiment by Category":       "category_grouped_bar",
        "🔥  Rating ↔ Sentiment Heatmap":  "rating_sentiment_heatmap",
        "📝  Top 20 Keywords":             "top_keywords",
        "☁️  Word Cloud — All":            "wc_all",
        "☁️  Word Cloud — Positive Only":  "wc_pos",
        "☁️  Word Cloud — Negative Only":  "wc_neg",
        "🌡️  Overall Sentiment Gauge":     "sentiment_gauge",
        "📦  Confidence Box Plot":         "confidence_boxplot",
        "🔵  Rating vs Confidence Scatter":"scatter_rating_confidence",
        "🌞  Sunburst Category→Sentiment": "sunburst_category_sentiment",
        "🎻  Word Count Violin":           "word_count_violin",
    }

    def __init__(self, raw_df, engine):
        self.raw_df  = raw_df
        self.engine  = engine
        self.res_df  = None
        self.viz     = None
        self._build()

    # ── Build all widgets ────────────────────────────────────────
    def _build(self):
        W = {"description_width": "150px"}

        # ── Control Panel widgets ──
        self.w_model = widgets.Dropdown(
            options=list(SentimentEngine.MODELS.keys()),
            value=list(SentimentEngine.MODELS.keys())[0],
            description="🧠 AI Model:",
            style=W, layout=widgets.Layout(width="100%"),
        )
        self.w_n = widgets.IntSlider(
            value=min(100, len(self.raw_df)), min=20,
            max=min(300, len(self.raw_df)), step=10,
            description="📊 Samples:",
            style=W, layout=widgets.Layout(width="100%"),
            continuous_update=False,
        )
        self.w_run = widgets.Button(
            description="🚀  Run Analysis",
            layout=widgets.Layout(width="180px", height="38px"),
            style={"button_color": "#667eea"},
        )
        self.w_prog = widgets.Output()

        # ── Chart Panel widgets ──
        self.w_chart = widgets.Dropdown(
            options=list(self.CHARTS.keys()),
            value=list(self.CHARTS.keys())[0],
            description="📈 Chart:",
            style=W, layout=widgets.Layout(width="100%"),
            disabled=True,
        )
        self.w_view = widgets.Button(
            description="👁️  View Chart",
            button_style="info",
            layout=widgets.Layout(width="160px", height="38px"),
            disabled=True,
        )
        self.w_all = widgets.Button(
            description="📋  All Charts",
            button_style="warning",
            layout=widgets.Layout(width="160px", height="38px"),
            disabled=True,
        )
        self.w_out = widgets.Output()
        self.w_metrics = widgets.Output()

        # ── Events ──
        self.w_run.on_click(self._run)
        self.w_view.on_click(self._view)
        self.w_all.on_click(self._all)

        # ── Layout ──
        sep = widgets.HTML("<hr style='border:none;border-top:1px solid #e0e4f0;margin:12px 0'>")

        ctrl = widgets.VBox([
            widgets.HTML('''<div style="font-family:'DM Sans',sans-serif;font-size:1.05em;
                         font-weight:700;color:#1a1a2e;margin-bottom:14px">
                         ⚙️  Control Panel</div>'''),
            self.w_model, self.w_n,
            widgets.HBox([self.w_run]),
            self.w_prog,
        ], layout=widgets.Layout(
            border="1px solid #e0e4f0", border_radius="14px",
            padding="22px", margin="8px 0", background_color="#fafbff",
        ))

        chart_panel = widgets.VBox([
            widgets.HTML('''<div style="font-family:'DM Sans',sans-serif;font-size:1.05em;
                         font-weight:700;color:#1a1a2e;margin-bottom:14px">
                         📈  Chart Explorer</div>'''),
            self.w_chart,
            widgets.HBox([self.w_view,
                          widgets.HTML("&nbsp;&nbsp;&nbsp;"),
                          self.w_all]),
            self.w_metrics,
            self.w_out,
        ], layout=widgets.Layout(
            border="1px solid #e0e4f0", border_radius="14px",
            padding="22px", margin="8px 0",
        ))

        header = widgets.HTML("""
        <div style="background:linear-gradient(135deg,#0f0c29,#302b63,#24243e);
                    padding:28px 32px;border-radius:16px;margin-bottom:14px;
                    box-shadow:0 16px 48px rgba(0,0,0,0.35);">
          <h2 style="font-family:'DM Sans',sans-serif;color:white;margin:0 0 8px;
                     font-size:1.7em;font-weight:800;letter-spacing:-0.4px">
            🤖 AI Review Intelligence Dashboard
          </h2>
          <p style="color:rgba(255,255,255,0.65);margin:0;font-size:0.9em">
            Step 1 — choose model &amp; sample size &nbsp;|&nbsp;
            Step 2 — click <b>Run Analysis</b> &nbsp;|&nbsp;
            Step 3 — explore charts
          </p>
        </div>
        """)

        self.ui = widgets.VBox([header, ctrl, chart_panel])

    # ── Event handlers ───────────────────────────────────────────
    def _run(self, _):
        with self.w_prog:
            clear_output(wait=True)
            n = self.w_n.value
            sample = (self.raw_df.sample(n, random_state=42)
                      if n < len(self.raw_df) else self.raw_df)
            print(f"⏳  Analysing {n} records with {self.w_model.value} …\n")
            try:
                self.res_df = self.engine.analyse(sample, self.w_model.value)
                self.viz    = VizEngine(self.res_df)

                pos = (self.res_df["sentiment"] == "POSITIVE").sum()
                neg = (self.res_df["sentiment"] == "NEGATIVE").sum()
                neu = (self.res_df["sentiment"] == "NEUTRAL").sum()
                avg_c = self.res_df["confidence"].mean()
                avg_r = self.res_df["rating"].mean()

                for w in [self.w_chart, self.w_view, self.w_all]:
                    w.disabled = False

                with self.w_metrics:
                    clear_output(wait=True)
                    display(HTML(f"""
                    <div style="margin:14px 0">
                    <div style="display:flex;gap:10px;flex-wrap:wrap">
                      <span class="pill pill-pos">😊 Positive: {pos} ({pos/n*100:.0f}%)</span>
                      <span class="pill pill-neg">😞 Negative: {neg} ({neg/n*100:.0f}%)</span>
                      <span class="pill pill-neu">😐 Neutral: {neu} ({neu/n*100:.0f}%)</span>
                      <span class="pill pill-info">🎯 Avg Confidence: {avg_c:.1%}</span>
                      <span class="pill pill-info">⭐ Avg Rating: {avg_r:.2f}</span>
                    </div>
                    <p style="font-size:0.82em;color:#888;margin:10px 0 0">
                    ✅ Analysis complete! Select a chart below and click <b>👁️ View Chart</b>.
                    </p>
                    </div>
                    """))
                print("✅  Done! Select a chart above.")

            except Exception as e:
                print(f"❌  Error: {e}")

    def _render(self, key):
        method = self.CHARTS[key]
        if method.startswith("wc_"):
            filt = {"wc_all": "ALL", "wc_pos": "POSITIVE", "wc_neg": "NEGATIVE"}[method]
            self.viz.word_cloud_fig(filt)
            plt.show()
        else:
            getattr(self.viz, method)().show()

    def _view(self, _):
        if self.res_df is None:
            with self.w_out:
                clear_output(wait=True)
                print("⚠️  Please run analysis first (Step 1 → 2).")
            return
        with self.w_out:
            clear_output(wait=True)
            key = self.w_chart.value
            print(f"🎨  Rendering: {key} …")
            try:
                self._render(key)
            except Exception as e:
                print(f"❌  Chart error: {e}")

    def _all(self, _):
        if self.res_df is None:
            with self.w_out:
                clear_output()
                print("⚠️  Please run analysis first.")
            return
        with self.w_out:
            clear_output(wait=True)
            print("🎨  Generating all charts — this may take a moment …\n")
            for key in self.CHARTS:
                print(f"  → {key}")
                try:
                    self._render(key)
                except Exception as e:
                    print(f"     ⚠️ Skipped: {e}")

    def show(self):
        display(self.ui)


# ── Instantiate and display ───────────────────────────────────
dash = Dashboard(RAW_DF, ENGINE)
dash.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — EXPORT RESULTS  (Optional)                        ║
# ╚══════════════════════════════════════════════════════════════╝

def export_results():
    """Export analysis results and static summary charts to files."""
    if dash.res_df is None:
        print("⚠️  Run analysis first (Cell 6 → 🚀 Run Analysis).")
        return

    df_out = dash.res_df.copy()
    df_out["date"] = df_out["date"].dt.strftime("%Y-%m-%d")

    # CSV export
    df_out.to_csv("sentiment_results.csv", index=False)
    print("✅  Saved: sentiment_results.csv")

    # Summary stats
    summary = df_out.groupby("sentiment").agg(
        count=("sentiment","count"),
        avg_confidence=("confidence","mean"),
        avg_rating=("rating","mean"),
    ).round(3)
    summary.to_csv("sentiment_summary.csv")
    print("✅  Saved: sentiment_summary.csv")

    # Static 4-panel chart PNG (matplotlib)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Sentiment Analysis — Summary Dashboard", fontsize=18, fontweight="bold", y=1.01)
    fig.patch.set_facecolor("white")

    # Panel 1 — Sentiment Pie
    cnt = df_out["sentiment"].value_counts()
    axes[0,0].pie(cnt.values, labels=cnt.index,
                  colors=[_C_MAP.get(s,"#aaa") for s in cnt.index],
                  autopct="%1.1f%%", startangle=90)
    axes[0,0].set_title("Sentiment Share", fontsize=13, fontweight="bold")

    # Panel 2 — Rating bars
    rc = df_out["rating"].round().astype(int).value_counts().sort_index()
    axes[0,1].bar([f"★{r}" for r in rc.index], rc.values,
                  color=["#FF4B6E","#FF8C42","#FFC107","#7CB9E8","#00C897"][:len(rc)])
    axes[0,1].set_title("Rating Distribution", fontsize=13, fontweight="bold")

    # Panel 3 — Trend
    dfs = df_out.sort_values("date")
    dfs["ma"] = dfs["score"].rolling(10, min_periods=1).mean()
    axes[1,0].plot(range(len(dfs)), dfs["ma"].values, color=_C_PRIMARY, linewidth=2)
    axes[1,0].axhline(0, color="#aaa", linestyle="--", alpha=0.5)
    axes[1,0].set_title("Sentiment Moving Average", fontsize=13, fontweight="bold")
    axes[1,0].set_xlabel("Review Index"); axes[1,0].set_ylabel("Score")

    # Panel 4 — Confidence histogram
    for sent, col in _C_MAP.items():
        sub = df_out[df_out["sentiment"] == sent]["confidence"]
        if len(sub) > 0:
            axes[1,1].hist(sub.values, bins=15, alpha=0.6, color=col, label=sent)
    axes[1,1].set_title("Confidence Distribution", fontsize=13, fontweight="bold")
    axes[1,1].legend()

    plt.tight_layout()
    plt.savefig("sentiment_dashboard.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅  Saved: sentiment_dashboard.png")

    # Download in Colab
    try:
        from google.colab import files
        for f in ["sentiment_results.csv", "sentiment_summary.csv", "sentiment_dashboard.png"]:
            files.download(f)
        print("✅  Downloads triggered!")
    except Exception:
        print("ℹ️  Run in Google Colab to auto-download files.")

export_results()
